# Stage 7：两组已验收7B的Colab闭环（准备稿，尚未执行）
沿用 `S6_Colab闭环.ipynb` 的 bf16 加载、LoRA合并和 `eval.colab_loop` 批处理机制。
只运行 `grpo_7b_repro`（nolacs）与 `comllm_7b_repro`（lacs），各6点：3台服务器、N=20/40/60、seed=0/1、1000槽、200槽热身、sequential，共12个唯一点。

两组闭环共同使用 **paper / search / greedy / max_new_tokens=16 / repetition_penalty=1.05**；S7开环是 **max15 / repetition_penalty=1.0**，不能将开环与闭环称为完全相同解码。
使用固定SFT目录，逐文件校验四片权重及元数据，并核对两组真实adapter SHA和2026-11-08 18:38香港时间的严格联合验收报告。

执行前将独立 `s7_closed_code/` 整个文件夹上传到 `MyDrive/COMLLM/s7_closed_code/`（code.zip、meta.json、sha256_manifest.json）。它由既定S6的 `CL.pack` 生成，含36个py/json成员；不要覆盖原S7训练包。
Drive应保留 `s7_runs/acceptance_20261108T103821.435715Z.json`。依赖沿用当前已验证Colab环境，本notebook不安装或卸载任何库。
全新输出根为 `MyDrive/COMLLM/s7_closed_runs_20261113_a100_standard/`。先完成CPU身份/公式golden校验，再加载GPU；没有主动开始运行本notebook之前，不会产生闭环结果。
续跑仅跳过完整点；provenance、硬件环境或已有行身份变化会拒绝继续。不要同时启动两个副本；报错即停，完成必须有最终12点验收报告。7B不转到Jetson。

## 1. 固定入口与现有依赖（不安装）

In [1]:
import csv, gc, hashlib, importlib.metadata as metadata, json, math, os, shutil, sys, time, zipfile
from datetime import datetime, timezone
from pathlib import Path
from google.colab import drive
for name, version in {'transformers':'5.20.2', 'peft':'0.21.2'}.items():
    assert metadata.version(name) == version, f'{name}版本漂移；停止并核对环境'
VERSIONS = {k: metadata.version(k) for k in ('torch','transformers','peft','accelerate','numpy')}
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/COMLLM')
CODE = DRIVE / 's7_closed_code'
WORK = Path('/content/comllm_s7_closed_20261108')
BASE_DRIVE = DRIVE / 'reference_model/qwen2.5-7b-mec-sft-final-all'
BASE = Path('/content/s7_sft7b')
TRAIN = DRIVE / 's7_runs'
OUT = DRIVE / 's7_closed_runs_20261113_a100_standard'
RUNS = {'grpo_7b_repro':'nolacs', 'comllm_7b_repro':'lacs'}
ACCEPTANCE_NAME = 'acceptance_20261108T103821.435715Z.json'
ACCEPTANCE_SHA = '28accfda57ff3a359f030ecd74d73bb789432cddd6bfc6ffabe9527388f2d92a'
CODE_SHA = '9b104f5a20994f4a315d88e1ec27604d7fde1838b0556f895ae4d8c4b3cd6f2c'
META_SHA = 'af213e865323a2efc54c7968e8008599af27f35b5f457f025600edc00d088dde'
MANIFEST_SHA = '7095bf765f8b4cb2ca6de8f909f095f0d2ff9819a8d8b990edc3d61df5dad2f4'
ADAPTER_SHA = {'nolacs':'4b5dc12b1300d91d18d5cc9cfeefe791ebe04fb579d1868f3741dab7d69c905e',
               'lacs':'dc1ce5def49209b34bb03fbf863c5ebe96a6aedda1a86a9ba4bfa7d1e5e51012'}
def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(8*1024*1024), b''): h.update(chunk)
    return h.hexdigest()
def canonical(value): return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',',':'), allow_nan=False)
def identity_sha(value): return hashlib.sha256(canonical(value).encode()).hexdigest()
def read_json(path): return json.loads(Path(path).read_text(encoding='utf-8'))
def write_new_json(path, value):
    with Path(path).open('x', encoding='utf-8') as f: json.dump(value, f, ensure_ascii=False, indent=2, allow_nan=False)
print('仅两组：', RUNS, '；新输出根：', OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
仅两组： {'grpo_7b_repro': 'nolacs', 'comllm_7b_repro': 'lacs'} ；新输出根： /content/drive/MyDrive/COMLLM/s7_closed_runs_20261113_a100_standard


## 2. 独立闭环包、全部成员与CPU公式golden
原S7训练包没有S6的独立meta与完整JSON依赖。这里核对独立小包；golden是S6的固定公式核对点，不依赖另一个golden数据文件。

In [2]:
assert sha(CODE / 'code.zip') == CODE_SHA, '闭环ZIP SHA变化'
assert sha(CODE / 'meta.json') == META_SHA, '闭环meta SHA变化'
assert sha(CODE / 'sha256_manifest.json') == MANIFEST_SHA, '闭环成员清单变化'
META, MANIFEST = read_json(CODE / 'meta.json'), read_json(CODE / 'sha256_manifest.json')
assert MANIFEST['archive_sha256'] == CODE_SHA and MANIFEST['meta_sha256'] == META_SHA
with zipfile.ZipFile(CODE / 'code.zip') as z:
    names = z.namelist()
    assert z.testzip() is None and len(names) == len(set(names)) == 36
    assert set(names) == set(META['files']) == set(MANIFEST['files'])
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in names)
    for name in names:
        payload, record = z.read(name), MANIFEST['files'][name]
        assert len(payload) == record['bytes'] and hashlib.sha256(payload).hexdigest() == record['sha256'], name
    WORK.mkdir(parents=True, exist_ok=True)
    z.extractall(WORK)
assert 'deploy/testbed.json' in names
for name in names: assert sha(WORK / name) == MANIFEST['files'][name]['sha256'], name
actual_members = {p.relative_to(WORK).as_posix() for p in WORK.rglob('*') if p.suffix in ('.py','.json') and '__pycache__' not in p.parts}
assert actual_members == set(names), '本地闭环解压目录有额外代码/JSON，拒绝混用'
for name in list(sys.modules):
    if name.split('.')[0] in ('eval','sim','deploy'): del sys.modules[name]
sys.path.insert(0, str(WORK))
os.chdir(WORK)
from eval import colab_loop as CL
assert Path(CL.__file__).resolve().is_relative_to(WORK.resolve())
assert CL.check_value() == META['check_mean_delay'] == 4.197929414659039
POINTS = {key: CL.points_for(key) for key in RUNS}
ALL_POINTS = [p for key in RUNS for p in POINTS[key]]
assert len(ALL_POINTS) == len({p['point'] for p in ALL_POINTS}) == 12
for key, points in POINTS.items():
    assert len(points) == 6 and {(p['n'],p['seed']) for p in points} == {(n,s) for n in (20,40,60) for s in (0,1)}
    assert all((p['servers'],p['slots'],p['warmup'],p['mode'],p['policy']) ==
               (3,1000,200,'sequential','llm:'+key) for p in points)
print('独立闭环包36成员通过；CPU golden=', META['check_mean_delay'], '；唯一点=',len(ALL_POINTS))

独立闭环包36成员通过；CPU golden= 4.197929414659039 ；唯一点= 12


## 3. 固定严格验收报告、两组adapter与训练身份（CPU读取）

In [3]:
acceptance_path = TRAIN / ACCEPTANCE_NAME
assert sha(acceptance_path) == ACCEPTANCE_SHA, '严格验收报告SHA变化'
ACCEPTANCE = read_json(acceptance_path)
assert ACCEPTANCE['status'] == ACCEPTANCE['verifier_result']['status'] == 'passed'
assert ACCEPTANCE['verifier_returncode'] == 0 and ACCEPTANCE['verifier_result']['paired_identity_matches']
ACCEPTED = ACCEPTANCE['verifier_result']['groups']
TRAIN_IDENTITY = ACCEPTED['nolacs']['identity']
assert ACCEPTED['lacs']['identity'] == TRAIN_IDENTITY == read_json(TRAIN / 'pair.json')
ARTIFACTS = {}
for key, group in RUNS.items():
    folder = TRAIN / group
    cfg, summary = read_json(folder/'config.json'), read_json(folder/'summary.json')
    assert cfg['group'] == group and cfg['lambda_lacs'] == (0.0 if group == 'nolacs' else 0.3)
    assert cfg['identity'] == TRAIN_IDENTITY
    assert summary['status'] == 'complete' and summary['training_complete'] is True
    assert summary['step'] == summary['target_steps'] == 208
    adapter = folder / 'lora'
    assert read_json(adapter/'adapter_config.json') == ACCEPTED[group]['adapter']['config']
    assert sha(adapter/'adapter_model.safetensors') == ADAPTER_SHA[group] == ACCEPTED[group]['adapter']['sha256']
    adapter_files = {p.name:{'bytes':p.stat().st_size,'sha256':sha(p)} for p in
                     (adapter/'adapter_config.json', adapter/'adapter_model.safetensors')}
    ARTIFACTS[key] = {'group':group,'lambda_lacs':cfg['lambda_lacs'], 'adapter_path':str(adapter),
                     'adapter_files':adapter_files,'training_identity':cfg['identity'],
                     'config_sha256':sha(folder/'config.json'),'summary_sha256':sha(folder/'summary.json')}
print('严格验收报告、两组实际adapter SHA、pair训练身份通过')

严格验收报告、两组实际adapter SHA、pair训练身份通过


## 4. 复用已验证本地SFT；缺失或不符时从固定Drive目录复制
四个分片及所有tokenizer/config文件都与真实验收报告中的SHA逐一一致才使用；不按hidden_size搜索目录。

In [4]:
EXPECTED_BASE = TRAIN_IDENTITY['base_model']['files']
assert sum(name.endswith('.safetensors') for name in EXPECTED_BASE) == 4
BASE.mkdir(parents=True, exist_ok=True)
invalid = [name for name,record in EXPECTED_BASE.items() if not (BASE/name).is_file()
           or (BASE/name).stat().st_size != record['bytes'] or sha(BASE/name) != record['sha256']]
assert shutil.disk_usage(BASE).free > sum(EXPECTED_BASE[n]['bytes'] for n in invalid) + 5*1024**3, '本地复制空间不足'
for name in invalid:
    record, source, target = EXPECTED_BASE[name], BASE_DRIVE/name, BASE/name
    assert source.is_file() and source.stat().st_size == record['bytes'], f'固定Drive基座文件缺失或大小错误：{name}'
    temporary = target.with_name(target.name+'.partial')
    shutil.copyfile(source, temporary)
    assert sha(temporary) == record['sha256'], f'固定Drive基座SHA错误：{name}'
    os.replace(temporary, target)
assert {p.name for p in BASE.glob('*.safetensors')} == {n for n in EXPECTED_BASE if n.endswith('.safetensors')}, '本地基座存在额外分片'
BASE_FILES = {name:{'bytes':(BASE/name).stat().st_size,'sha256':sha(BASE/name)} for name in EXPECTED_BASE}
assert BASE_FILES == EXPECTED_BASE
BASE_GENERATION = read_json(BASE/'generation_config.json')
assert BASE_GENERATION['repetition_penalty'] == 1.05, 'S6既定重复惩罚变化'
DECODE = {'format':'paper','parse':'search','do_sample':False,'num_beams':1,
          'max_new_tokens':16,'repetition_penalty':1.05,'eos_token':'<|im_end|>',
          'inherited_generation_config':BASE_GENERATION}
print('固定SFT全部文件SHA通过；复用', len(EXPECTED_BASE)-len(invalid), '文件，复制',len(invalid),'文件')

固定SFT全部文件SHA通过；复用 13 文件，复制 0 文件


## 5. 保存provenance并保护续跑身份
CSV沿用S6旧列名 `gguf_sha256`，其中写入实际base+adapter文件SHA清单的联合SHA；此处没有生成GGUF。完整文件SHA、训练身份和验收报告保存在provenance。

In [5]:
MODEL_SHA = {key:identity_sha({'base_files':BASE_FILES,'adapter_files':ARTIFACTS[key]['adapter_files']}) for key in RUNS}
IDENTITY = {'schema_version':1, 'run_order':list(RUNS), 'points':ALL_POINTS, 'decode':DECODE,
    'open_loop_decode':{'max_new_tokens':15,'repetition_penalty':1.0}, 'versions':VERSIONS,
    'code':{'archive_sha256':CODE_SHA,'meta_sha256':META_SHA,'manifest_sha256':MANIFEST_SHA,'files':MANIFEST['files']},
    'base_source':str(BASE_DRIVE),'base_used':str(BASE),'base_files':BASE_FILES,
    'artifacts':ARTIFACTS,'model_identity_sha256':MODEL_SHA,
    'acceptance':{'path':str(acceptance_path),'sha256':ACCEPTANCE_SHA,'report':ACCEPTANCE},
    'legacy_gguf_sha256_column':'SHA256 of canonical base_files+adapter_files manifest; bf16 merged LoRA, no GGUF file'}
OUT.mkdir(parents=True, exist_ok=True)
provenance_path, results_path = OUT/'provenance.json', OUT/'results.csv'
if provenance_path.exists():
    assert read_json(provenance_path)['identity'] == IDENTITY, '闭环身份变化，拒绝续跑；必须使用独立输出根'
else:
    assert not list(OUT.iterdir()), '输出根已有证据但缺少provenance，拒绝混入'
    write_new_json(provenance_path, {'created_utc':datetime.now(timezone.utc).isoformat(),'identity':IDENTITY})
EXPECTED_POINTS = {p['point']:p for p in ALL_POINTS}
def checked_rows(require_all=False):
    rows = []
    if results_path.exists():
        with results_path.open(newline='',encoding='utf-8') as f:
            reader = csv.DictReader(f, strict=True)
            assert reader.fieldnames == CL.FIELDS, '闭环CSV列变化'
            rows = list(reader)
    names = [r['point'] for r in rows]
    assert len(names) == len(set(names)) and set(names) <= set(EXPECTED_POINTS), '重复或未知闭环点'
    for row in rows:
        assert None not in row and all(value is not None for value in row.values()), '闭环CSV缺字段或有额外列'
        p = EXPECTED_POINTS[row['point']]
        for field in ('servers','n','seed','slots','warmup'): assert int(row[field]) == p[field], (row['point'],field)
        for field in ('mode','policy'): assert row[field] == p[field], (row['point'],field)
        key = row['policy'].removeprefix('llm:')
        assert row['gguf_sha256'] == MODEL_SHA[key], '闭环行模型身份变化'
        for field in ('mean_delay','p95_delay','drop_rate','local_frac','top_server_frac','jain','pred_gap','agree_greedy','ms_per_decision','wall_s'):
            assert math.isfinite(float(row[field])), (row['point'],field)
        assert int(row['n_decisions']) > 0 and 0 <= int(row['fallback']) <= int(row['n_decisions'])
        assert int(row['n_tasks']) > 0
        shares = [float(x) for x in row['server_share'].split(';')]
        assert len(shares) == p['servers'] and all(math.isfinite(x) and 0 <= x <= 1 for x in shares)
        assert abs(sum(shares)) < 1e-9 or abs(sum(shares)-1) <= 0.0003, '服务器份额合计异常'
        for field in ('drop_rate','local_frac','top_server_frac','jain','agree_greedy'): assert 0 <= float(row[field]) <= 1
    if require_all: assert set(names) == set(EXPECTED_POINTS) and len(rows) == 12, '12点尚未全部完成'
    return rows
existing = checked_rows()
print('provenance身份通过；已完成',len(existing),'/12点；仅跳过这些完整点')

provenance身份通过；已完成 6 /12点；仅跳过这些完整点


## 6. GPU加载并依次执行两组（这是实际执行入口，准备阶段未运行）
沿用S6：SFT bf16 → 合并指定LoRA → `make_paper_decider` → `run_points`。每组结束强制核对6点；线程失败或缺点会停止。断线只保存已完整写入CSV的点，重新全部运行可在同一身份下继续。

In [6]:
def checked_generation_settings(generation_config):
    defaults = generation_config._get_default_generation_params()
    raw = {name: getattr(generation_config, name)
           for name in ('num_beams', 'repetition_penalty')}
    effective = {name: defaults[name] if value is None else value
                 for name, value in raw.items()}
    print('生成配置检查：', {'raw': raw, 'effective': effective}, flush=True)
    beams, penalty = effective['num_beams'], effective['repetition_penalty']
    assert type(beams) is int and beams == 1, (
        f"num_beams不符：raw={raw['num_beams']!r}, effective={beams!r}")
    assert type(penalty) in (int, float) and penalty == 1.05, (
        f"repetition_penalty不符：raw={raw['repetition_penalty']!r}, effective={penalty!r}")
    return {'raw': raw, 'effective': effective}


assert read_json(provenance_path)['identity'] == IDENTITY, '执行前provenance身份变化'
for key in RUNS:
    adapter = Path(ARTIFACTS[key]['adapter_path'])
    for name, record in ARTIFACTS[key]['adapter_files'].items(): assert sha(adapter/name) == record['sha256'], '执行前adapter身份变化'
pending = set(EXPECTED_POINTS) - {r['point'] for r in checked_rows()}
if pending:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), '需要bf16 CUDA GPU'
    hardware = {'gpu':torch.cuda.get_device_name(0),'versions':VERSIONS,'bf16':True}
    hardware_path = OUT/'runtime_identity.json'
    if hardware_path.exists(): assert read_json(hardware_path) == hardware, '运行硬件/版本变化，拒绝续跑'
    else: write_new_json(hardware_path, hardware)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ')
    write_new_json(OUT/f'invocation_{stamp}.json', {'started_utc':datetime.now(timezone.utc).isoformat(),
        'hardware':hardware,'provenance_sha256':sha(provenance_path),'pending_points':sorted(pending),'decode':DECODE})
    for key in RUNS:
        done = {r['point'] for r in checked_rows()}
        if {p['point'] for p in POINTS[key]} <= done:
            print(key,'6点已完成，跳过模型加载'); continue
        t0 = time.monotonic()
        tok = AutoTokenizer.from_pretrained(str(BASE), local_files_only=True, trust_remote_code=False)
        tok.padding_side = 'left'
        assert tok.pad_token_id is not None
        base = AutoModelForCausalLM.from_pretrained(str(BASE), local_files_only=True, trust_remote_code=False,
                                                    dtype=torch.bfloat16).to('cuda')
        model = PeftModel.from_pretrained(base, ARTIFACTS[key]['adapter_path'], local_files_only=True).merge_and_unload()
        del base
        try:
            model.eval()
            checked_generation_settings(model.generation_config)
            decide = CL.make_paper_decider(model,tok,'cuda',max_new_tokens=16,parse='search')
            CL.run_points(POINTS[key],decide,str(results_path),sha=MODEL_SHA[key])
            assert {p['point'] for p in POINTS[key]} <= {r['point'] for r in checked_rows()}, f'{key}闭环缺点，停止'
            print(key,'6点完成；用时分钟',round((time.monotonic()-t0)/60,2))
        finally:
            if 'decide' in locals(): del decide
            del model, tok
            gc.collect()
            torch.cuda.empty_cache()
else:
    print('12点已完整存在；无需GPU加载')

grpo_7b_repro 6点已完成，跳过模型加载


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

生成配置检查： {'raw': {'num_beams': None, 'repetition_penalty': 1.05}, 'effective': {'num_beams': 1, 'repetition_penalty': 1.05}}
llm_comllm_7b_repro_k3_n20_sequential_s0_t1000   平均延迟  4.86 槽  丢弃   6.2%  和单步最优一致   0.2%  回退 0
llm_comllm_7b_repro_k3_n20_sequential_s1_t1000   平均延迟  5.02 槽  丢弃   6.3%  和单步最优一致   0.2%  回退 0
llm_comllm_7b_repro_k3_n40_sequential_s0_t1000   平均延迟  6.86 槽  丢弃  55.6%  和单步最优一致   0.1%  回退 0
llm_comllm_7b_repro_k3_n40_sequential_s1_t1000   平均延迟  6.76 槽  丢弃  54.6%  和单步最优一致   0.0%  回退 0
llm_comllm_7b_repro_k3_n60_sequential_s0_t1000   平均延迟  7.17 槽  丢弃  84.1%  和单步最优一致   0.0%  回退 0
llm_comllm_7b_repro_k3_n60_sequential_s1_t1000   平均延迟  6.92 槽  丢弃  83.8%  和单步最优一致   0.0%  回退 0
共 71766 次决策，17993 批（平均每批 4.0 个），GPU 每次决策 128.2 ms，用时 153.8 分钟
comllm_7b_repro 6点完成；用时分钟 153.94


## 7. 最终12点验收与结果摘要
仅有训练验收、provenance或模型加载成功都不代表闭环完成。需要12条精确点名、参数与模型身份全部匹配、指标有限，才写入新的闭环验收报告；不把准备稿称为闭环实测。

In [7]:
assert read_json(provenance_path)['identity'] == IDENTITY, '最终验收前provenance身份变化'
rows = checked_rows(require_all=True)
aggregate = {}
for key in RUNS:
    aggregate[key] = {}
    for n in (20,40,60):
        selected = [r for r in rows if r['policy'] == 'llm:'+key and int(r['n']) == n]
        assert len(selected) == 2 and {int(r['seed']) for r in selected} == {0,1}
        aggregate[key][str(n)] = {'seeds':2,'mean_delay':sum(float(r['mean_delay']) for r in selected)/2,
            'drop_rate':sum(float(r['drop_rate']) for r in selected)/2,
            'fallback':sum(int(r['fallback']) for r in selected),
            'agree_greedy':sum(float(r['agree_greedy']) for r in selected)/2}
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ')
final = {'status':'complete','checked_utc':datetime.now(timezone.utc).isoformat(),'unique_points':12,
         'results_sha256':sha(results_path),'provenance_sha256':sha(provenance_path),
         'training_acceptance_sha256':ACCEPTANCE_SHA,'code_archive_sha256':CODE_SHA,
         'runtime':read_json(OUT/'runtime_identity.json'),'model_identity_sha256':MODEL_SHA,
         'decode':DECODE,'aggregate':aggregate,'rows':rows}
final_path = OUT/f'closed_acceptance_{stamp}.json'
write_new_json(final_path,final)
print('两组7B闭环12点验收通过：',final_path)
print(json.dumps(aggregate,ensure_ascii=False,indent=2))

两组7B闭环12点验收通过： /content/drive/MyDrive/COMLLM/s7_closed_runs_20261113_a100_standard/closed_acceptance_20261113T092611.183871Z.json
{
  "grpo_7b_repro": {
    "20": {
      "seeds": 2,
      "mean_delay": 4.91693,
      "drop_rate": 0.06143,
      "fallback": 0,
      "agree_greedy": 0.00109
    },
    "40": {
      "seeds": 2,
      "mean_delay": 6.85047,
      "drop_rate": 0.531505,
      "fallback": 0,
      "agree_greedy": 0.000375
    },
    "60": {
      "seeds": 2,
      "mean_delay": 7.296825,
      "drop_rate": 0.840975,
      "fallback": 0,
      "agree_greedy": 0.00028
    }
  },
  "comllm_7b_repro": {
    "20": {
      "seeds": 2,
      "mean_delay": 4.937754999999999,
      "drop_rate": 0.062485,
      "fallback": 0,
      "agree_greedy": 0.00167
    },
    "40": {
      "seeds": 2,
      "mean_delay": 6.808345,
      "drop_rate": 0.550665,
      "fallback": 0,
      "agree_greedy": 0.00046
    },
    "60": {
      "seeds": 2,
      "mean_delay": 7.04481,
      "drop_rate": 

结果留在独立 `s7_closed_runs_20261113_a100_standard/`，需要下载 `results.csv`、`provenance.json`、`runtime_identity.json`、调用记录与最终 `closed_acceptance_*.json` 一同复核。
本notebook没有操作Jetson、没有训练，也不会覆盖S7训练包或训练结果。GPU运行完成后由执行者断开并删除Colab运行时。